# Asset Class Trend Following (equity-2MA) 策略回測

## 1. 策略說明
本策略採用 Asset Class Trend Following 方法，針對 131 檔商品進行回測。核心邏輯結合了雙移動平均線 (Double SMA) 的趨勢過濾、ROC 動能指標，以及多重流動性與價格趨勢濾網。策略旨在最大化風險調整後報酬（Calmar Ratio），並透過螞蟻演算法（ACO）對參數進行全域最佳化。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backtest_equity2MA import clean_data, Backtester, calculate_metrics

# 2. 資料清理與預處理
data_file = '樣本集-1.xlsx'
prices, volumes, code_to_name = clean_data(data_file)
print(f"資料載入完成: {len(prices)} 交易日, {len(code_to_name)} 檔商品")

## 3. 最佳化參數設定
在此區塊集中設定經過多輪最佳化後的最終參數，以利後續修正使用。此組參數在 2024-2025 測試期間之 Calmar Ratio 接近 4。

In [ ]:
SMA1 = 310
SMA2 = 290
ROC_PERIOD = 14
STOP_LOSS_TYPE = 'ma'  # 'peak' 或 'ma'
STOP_LOSS_VAL = 60     # peak 模式下為比例 (0.01-0.10)，ma 模式下為天數 (5-60)
REBALANCE_INTERVAL = 10
INITIAL_CAPITAL = 30000000

## 4. 執行策略回測
使用 `Backtester` 引擎執行回測邏輯，包含動態資金分配、多重濾網篩選與停損機制。

In [ ]:
bt = Backtester(prices, volumes, code_to_name, initial_capital=INITIAL_CAPITAL)
eq, trades, hold, trades2, daily = bt.run(
    SMA1, SMA2, ROC_PERIOD, STOP_LOSS_VAL, REBALANCE_INTERVAL, STOP_LOSS_TYPE
)

cagr, mdd, calmar, total_ret = calculate_metrics(eq)
print(f"全週期 (2019-2025) 績效總結:")
print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"最大回撤 (MaxDD): {mdd:.2%}")
print(f"Calmar Ratio: {calmar:.2f}")

## 5. 分段績效評估
分別查看樣本集 (2019-2023) 與測試集 (2024-2025) 的表現。

In [ ]:
eq_tr = eq[eq['日期'] <= '2023-12-31']
c_tr, m_tr, cal_tr, _ = calculate_metrics(eq_tr)
print(f"樣本集 (2019-2023): CAGR={c_tr:.2%}, MDD={m_tr:.2%}, Calmar={cal_tr:.2f}")

eq_te = eq[eq['日期'] >= '2024-01-02'].copy()
if not eq_te.empty:
    iv = eq_te['權益值'].iloc[0]
    eq_te['權益值'] = eq_te['權益值'] / iv * 30000000
    p = eq_te['權益值'].cummax()
    eq_te['回撤(Drawdown)'] = (eq_te['權益值'] - p) / p
    c_te, m_te, cal_te, _ = calculate_metrics(eq_te)
    print(f"測試集 (2024-2025): CAGR={c_te:.2%}, MDD={m_te:.2%}, Calmar={cal_te:.2f}")

## 6. 視覺化權益曲線
繪製策略之權益曲線與回撤圖表。

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(eq['日期'], eq['權益值'], label='Equity Curve')
plt.title('Strategy Performance (2019-2025)')
plt.legend()
plt.subplot(2, 1, 2)
plt.fill_between(eq['日期'], eq['回撤(Drawdown)'], 0, color='red', alpha=0.3)
plt.title('Drawdown Chart')
plt.tight_layout()
plt.show()

## 7. 輸出成果報表
將回測結果輸出至 Excel 檔案，包含詳細交易紀錄與各項績效指標。

In [ ]:
win_rate = (trades2['損益'] > 0).mean() if not trades2.empty else 0
summary_df = pd.DataFrame([{
    '策略名稱': 'Asset Class Trend Following (equity-2MA)',
    '參數': f'SMA:({SMA1},{SMA2}), ROC:{ROC_PERIOD}, SL:{STOP_LOSS_TYPE}({STOP_LOSS_VAL}), Rebalance:{REBALANCE_INTERVAL}',
    'CAGR': f'{cagr:.2%}',
    'MaxDD': f'{mdd:.2%}',
    'Calmar Ratio': f'{calmar:.2f}',
    '勝率': f'{win_rate:.2%}',
    '總報酬率': f'{total_ret:.2%}'
}])

output_file = 'trendstrategy_results_equity2MA.xlsx'
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    trades.to_excel(writer, sheet_name='Trades', index=False)
    trades2.to_excel(writer, sheet_name='Trades2', index=False)
    eq.to_excel(writer, sheet_name='Equity_Curve', index=False)
    hold.to_excel(writer, sheet_name='Equity_Hold', index=False)
    daily.to_excel(writer, sheet_name='Daily', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
print(f"報表已產出至: {output_file}")